# Chapter 10, Exercise 3: Intent and slot annotation in Label Studio with agreement

> **How to use this notebook.** Open it in Google Colab (File > Upload notebook, or the Colab badge on the companion website), then run the cells from top to bottom. Runtime > Change runtime type lets you pick a GPU when one is recommended below. Everything else runs on the free CPU tier.

## The exercise

**Chapter 10, Exercise 3.** Using TARIC-SLU, or a small licensed set of task-oriented Arabic speech commands you record or select, annotate 20 utterances in Label Studio with an intent-and-slot ontology, export the structured JSON, and compute a baseline inter-annotator agreement; then state which metric you would use to evaluate intent detection, slot filling, and dialogue state tracking. If you use SADA, first verify that the selected utterances suit the ontology rather than assuming they are task-oriented queries.

**Note on data.** TARIC-SLU is distributed from the authors' site (CC BY-NC 4.0, download form). To keep the notebook self-contained it uses **Speech-MASSIVE** (`FBK-MT/Speech-MASSIVE`, Arabic `ar-SA` split, CC BY-NC-SA 4.0; human-recorded task-oriented commands with intent and slot annotations, Section 10.9), which is exactly the kind of licensed command set the exercise allows. Twenty utterances are selected, the Label Studio project is generated, and the agreement is computed between the corpus's gold labels and a second annotation. SADA is deliberately **not** used: its segments are broadcast conversation, not task-oriented queries, so they do not suit an intent-and-slot ontology; the notebook shows how to verify that.

## Requirements

No GPU is required; the notebook runs on Colab's free CPU runtime.

## Testing status

Data selection, project generation and the agreement computation were executed; the second annotation is simulated and labelled as such. The Label Studio application itself was not run.


<a href="https://colab.research.google.com/github/arabic-speech-book/arabic-speech-book.github.io/blob/main/docs/solutions/Chapter_10_Exercise_03.ipynb" target="_blank" rel="noopener"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

In [1]:
!pip install -q pandas pyarrow huggingface_hub soundfile scikit-learn

## 1. Select 20 utterances and define the ontology

Speech-MASSIVE inherits MASSIVE's 60 intents and 55 slot types. For a small annotation exercise we restrict to a compact ontology of **10 intents** and **8 slot types** drawn from four scenarios (transport, calendar, alarm, weather and datetime), and select 20 utterances whose gold intent falls in that set.

In [2]:
import io, json, os, numpy as np, pandas as pd, pyarrow.parquet as pq, soundfile as sf
from huggingface_hub import hf_hub_download

p = hf_hub_download("FBK-MT/Speech-MASSIVE", "ar-SA/train_115-00000-of-00001.parquet", repo_type="dataset")
pf = pq.ParquetFile(p)
tab = pf.read().to_pandas()

INTENTS = ["transport_ticket", "transport_taxi", "transport_query", "transport_traffic", "calendar_set", "calendar_query",
           "alarm_set", "alarm_query", "weather_query", "datetime_query"]
SLOTS = ["place_name", "date", "time", "transport_type", "event_name", "timeofday", "general_frequency", "time_zone"]
sel = tab[tab["intent_str"].isin(INTENTS)].head(20).reset_index(drop=True)
print(len(sel), "utterances selected;", sel["intent_str"].value_counts().to_dict())
os.makedirs("slu_audio", exist_ok=True)
for i, r in sel.iterrows():
    y, sr = sf.read(io.BytesIO(r["audio"]["bytes"])); sf.write(f"slu_audio/utt_{i:02d}.wav", y, sr)
sel[["utt", "annot_utt", "intent_str"]]

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


20 utterances selected; {'calendar_query': 3, 'weather_query': 3, 'alarm_set': 3, 'transport_taxi': 3, 'transport_query': 2, 'calendar_set': 2, 'transport_ticket': 2, 'transport_traffic': 1, 'datetime_query': 1}


,utt,annot_utt,intent_str
0,هل فيه أي مواعيد اليوم في التقويم حقي,هل فيه أي مواعيد [date : اليوم] في التقويم حقي,calendar_query
1,أقرب محطة قطار,أقرب [place_name : محطة قطار],transport_query
2,كيف سيكون الطقس بكرة,كيف سيكون الطقس [date : بكرة],weather_query
3,كيف الطقس اليوم,كيف الطقس [date : اليوم],weather_query
4,ذكرني أدفع الإيجار كل شهر,ذكرني أدفع [event_name : الإيجار] [general_fre...,calendar_set
5,هل سوف تثلج هنا هذي العطلة هذا الأسبوع,هل سوف [weather_descriptor : تثلج] هنا [date :...,weather_query
6,نبهني الساعة عشرة الصبح لو سمحت,نبهني الساعة [time : عشرة الصبح] لو سمحت,alarm_set
7,سيارة المطار,[transport_type : سيارة] [place_name : المطار],transport_taxi
8,ابيك تحط منبه,ابيك تحط منبه,alarm_set
9,الاتصال سيارة,الاتصال [transport_type : سيارة],transport_taxi


### Verifying suitability (the SADA caution)

An utterance suits an intent-and-slot ontology only if it is a *request addressed to a system* with a recoverable goal. The cell below runs a quick check that every selected utterance has a gold intent from the ontology and at least one gold slot or a clear command verb. If you substitute SADA segments, the same check will show that most are narrative or conversational turns with no intent at all, and they should be excluded rather than force-labelled.

In [3]:
import re
def gold_slots(annot):
    return [(m.group(1).strip(), m.group(2).strip()) for m in re.finditer(r"\[([^:\]]+) : ([^\]]+)\]", annot)]
sel["gold_slots"] = sel["annot_utt"].map(gold_slots)
sel["suitable"] = sel.apply(lambda r: r["intent_str"] in INTENTS and (len(r["gold_slots"]) > 0 or len(r["utt"].split()) <= 3), axis=1)
print("suitable for the ontology:", int(sel["suitable"].sum()), "of", len(sel))
sel[["utt", "intent_str", "gold_slots", "suitable"]]

suitable for the ontology: 20 of 20


,utt,intent_str,gold_slots,suitable
0,هل فيه أي مواعيد اليوم في التقويم حقي,calendar_query,"[(date, اليوم)]",True
1,أقرب محطة قطار,transport_query,"[(place_name, محطة قطار)]",True
2,كيف سيكون الطقس بكرة,weather_query,"[(date, بكرة)]",True
3,كيف الطقس اليوم,weather_query,"[(date, اليوم)]",True
4,ذكرني أدفع الإيجار كل شهر,calendar_set,"[(event_name, الإيجار), (general_frequency, كل...",True
5,هل سوف تثلج هنا هذي العطلة هذا الأسبوع,weather_query,"[(weather_descriptor, تثلج), (date, هذي العطلة)]",True
6,نبهني الساعة عشرة الصبح لو سمحت,alarm_set,"[(time, عشرة الصبح)]",True
7,سيارة المطار,transport_taxi,"[(transport_type, سيارة), (place_name, المطار)]",True
8,ابيك تحط منبه,alarm_set,[],True
9,الاتصال سيارة,transport_taxi,"[(transport_type, سيارة)]",True


## 2. Generate the Label Studio project

Each task shows the audio and the transcript; the annotator chooses one intent and highlights slot spans in the transcript. The export is the standard Label Studio JSON, which the next section parses.

In [4]:
tasks = [{"data": {"audio": f"slu_audio/utt_{i:02d}.wav", "text": r["utt"], "id": int(i)}} for i, r in sel.iterrows()]
json.dump(tasks, open("slu_tasks.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
config = f'''<View>
  <Audio name="audio" value="$audio"/>
  <Header value="Transcript (highlight slot values in the text)"/>
  <Labels name="slots" toName="text">
    {''.join(f'<Label value="{s}"/>' for s in SLOTS)}
  </Labels>
  <Text name="text" value="$text"/>
  <Header value="Intent"/>
  <Choices name="intent" toName="text" choice="single" showInline="true">
    {''.join(f'<Choice value="{i}"/>' for i in INTENTS)}<Choice value="other"/>
  </Choices>
</View>'''
open("slu_config.xml", "w").write(config)
print("Label Studio: new project > Labeling Interface > Code: paste slu_config.xml > Import slu_tasks.json (serve slu_audio/ as local files).")
print(config)

Label Studio: new project > Labeling Interface > Code: paste slu_config.xml > Import slu_tasks.json (serve slu_audio/ as local files).
<View>
  <Audio name="audio" value="$audio"/>
  <Header value="Transcript (highlight slot values in the text)"/>
  <Labels name="slots" toName="text">
    <Label value="place_name"/><Label value="date"/><Label value="time"/><Label value="transport_type"/><Label value="event_name"/><Label value="timeofday"/><Label value="general_frequency"/><Label value="time_zone"/>
  </Labels>
  <Text name="text" value="$text"/>
  <Header value="Intent"/>
  <Choices name="intent" toName="text" choice="single" showInline="true">
    <Choice value="transport_ticket"/><Choice value="transport_taxi"/><Choice value="transport_query"/><Choice value="transport_traffic"/><Choice value="calendar_set"/><Choice value="calendar_query"/><Choice value="alarm_set"/><Choice value="alarm_query"/><Choice value="weather_query"/><Choice value="datetime_query"/><Choice value="other"/>
  </

## 3. Parse an export and compute agreement

`parse_export()` reads a Label Studio JSON export into (intent, {slot: value}) per task. To make the notebook runnable without a live annotation, a **second annotation is simulated** from the gold labels with a few deliberate disagreements (one intent confusion between the two transport intents, two slot-type confusions between `date` and `timeofday`, one missed slot). When you have your own export, replace `ann_B` with `parse_export("your_export.json")`.

In [5]:
def parse_export(path):
    out = {}
    for task in json.load(open(path, encoding="utf-8")):
        tid = task["data"]["id"]; intent, slots = "other", {}
        for ann in task.get("annotations", []):
            for res in ann.get("result", []):
                if res["from_name"] == "intent": intent = res["value"]["choices"][0]
                elif res["from_name"] == "slots":
                    slots[res["value"]["labels"][0]] = res["value"]["text"]
        out[tid] = (intent, slots)
    return out

ann_A = {int(i): (r["intent_str"], {s: v for s, v in r["gold_slots"] if s in SLOTS}) for i, r in sel.iterrows()}  # annotator A = corpus gold
# SIMULATED annotator B (illustrative): copy A and inject disagreements
import copy
ann_B = copy.deepcopy(ann_A)
ids = sorted(ann_B)
if len(ids) >= 4:
    i0 = next((i for i in ids if ann_B[i][0] == "transport_ticket"), ids[0]); ann_B[i0] = ("transport_taxi", ann_B[i0][1])
    changed = 0
    for i in ids:
        if "date" in ann_B[i][1] and changed < 2:
            v = ann_B[i][1].pop("date"); ann_B[i][1]["timeofday"] = v; changed += 1
    for i in ids:
        if len(ann_B[i][1]) >= 1 and i != i0:
            ann_B[i][1].pop(next(iter(ann_B[i][1]))); break

from sklearn.metrics import cohen_kappa_score
ia = [ann_A[i][0] for i in ids]; ib = [ann_B[i][0] for i in ids]
print(f"Intent: raw agreement = {np.mean([a == b for a, b in zip(ia, ib)]):.2f}, Cohen's kappa = {cohen_kappa_score(ia, ib):.3f}")

# slot agreement as pairwise F1 over (slot type, value) pairs, treating A as the reference
tp = fp = fn = 0
for i in ids:
    A = set(ann_A[i][1].items()); B = set(ann_B[i][1].items())
    tp += len(A & B); fp += len(B - A); fn += len(A - B)
P = tp/(tp+fp) if tp+fp else 0; R = tp/(tp+fn) if tp+fn else 0; F1 = 2*P*R/(P+R) if P+R else 0
print(f"Slots: pairwise P = {P:.2f}, R = {R:.2f}, F1 = {F1:.2f}  (exact match on type and value, A as reference)")
print("NOTE: annotator B is simulated in this run; replace with a real export.")

Intent: raw agreement = 0.95, Cohen's kappa = 0.943
Slots: pairwise P = 0.95, R = 0.91, F1 = 0.93  (exact match on type and value, A as reference)
NOTE: annotator B is simulated in this run; replace with a real export.


## 4. Which metric for which task (Section 10.9)

| Task | Metric | Matching rule to state |
|---|---|---|
| Intent detection | **intent accuracy** = correct utterances / all utterances (Eq. 10.1); for an imbalanced intent set add macro-F1 | single label per utterance |
| Slot filling | **slot F1** over (type, value) pairs (Eq. 10.2): P = TP/(TP+FP), R = TP/(TP+FN) | say whether values are compared as raw spans or after normalization (Table 10.5: الرياض vs Riyadh, غدا vs غدًا, ٢ vs 2), and whether partial overlap counts (SLU-F1) |
| Dialogue state tracking | **joint goal accuracy** = turns whose complete state matches exactly / all turns (Eq. 10.3) | one wrong, missing or extra slot value makes the whole turn wrong; report per-slot accuracy beside it |

For the agreement study itself, Cohen's kappa is the right statistic for the intent label (two annotators, categorical), and pairwise slot F1 with a stated reference direction and normalization for the spans; both must be reported with the sample size and the ontology version.